# Chapter 3: Coding Attention Mechanisms

This notebook implements attention mechanisms from scratch:
- Scaled dot-product attention
- Multi-head attention
- Causal masking for autoregressive models
- Visualization of attention patterns

## 3.1: Scaled Dot-Product Attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Mathematical foundation of attention
print("Scaled Dot-Product Attention Formula:")
print()
print("Attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V")
print()
print("Where:")
print("  Q (Query): What are we looking for?")
print("  K (Key): What patterns are available?")
print("  V (Value): What information to extract?")
print("  d_k: Dimension of keys (for scaling)")

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """Scaled dot-product attention mechanism"""
    
    def __init__(self, d_k, dropout=0.1):
        super().__init__()
        self.d_k = d_k
        self.dropout = nn.Dropout(p=dropout)
    
    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q: (batch, seq_len, d_k)
            K: (batch, seq_len, d_k)
            V: (batch, seq_len, d_v)
            mask: (batch, 1, seq_len) or (batch, seq_len, seq_len)
        
        Returns:
            output: (batch, seq_len, d_v)
            attention_weights: (batch, seq_len, seq_len)
        """
        # 1. Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # Shape: (batch, seq_len, seq_len)
        
        # 2. Apply mask (for causal attention or padding)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # 3. Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)
        
        # 4. Apply dropout
        attention_weights = self.dropout(attention_weights)
        
        # 5. Apply attention to values
        output = torch.matmul(attention_weights, V)
        # Shape: (batch, seq_len, d_v)
        
        return output, attention_weights

print("ScaledDotProductAttention class implemented")

## 3.2: Understanding Attention Weights

In [ ]:
# Example: Simple attention computation
batch_size = 2
seq_len = 4
d_k = 8
d_v = 8

# Create random Q, K, V
torch.manual_seed(42)
Q = torch.randn(batch_size, seq_len, d_k, device=device)
K = torch.randn(batch_size, seq_len, d_k, device=device)
V = torch.randn(batch_size, seq_len, d_v, device=device)

# Compute attention
attention = ScaledDotProductAttention(d_k, dropout=0.0)
output, weights = attention(Q, K, V, mask=None)

print(f"Input shapes:")
print(f"  Q: {Q.shape}")
print(f"  K: {K.shape}")
print(f"  V: {V.shape}")
print(f"\nOutput shapes:")
print(f"  output: {output.shape}")
print(f"  attention_weights: {weights.shape}")
print(f"\nAttention weights for first sample, first query position:")
print(weights[0, 0, :])  # How much each position is attended to
print(f"\nSum of weights (should be 1.0): {weights[0, 0, :].sum().item():.4f}")

## 3.3: Causal Masking for Autoregressive Models

In [ ]:
def create_causal_mask(seq_len, device):
    """
    Create a causal mask to prevent attending to future positions.
    
    For a sequence of length 4:
    [[1, 0, 0, 0],
     [1, 1, 0, 0],
     [1, 1, 1, 0],
     [1, 1, 1, 1]]
    """
    mask = torch.ones(seq_len, seq_len, device=device)
    mask = torch.triu(mask, diagonal=1)  # Upper triangular = 1
    mask = (1 - mask).bool()  # Flip: 0->1 (valid), 1->0 (masked)
    return mask

# Example
seq_len = 4
causal_mask = create_causal_mask(seq_len, device)
print(f"Causal mask (1 = attend, 0 = mask):")
print(causal_mask.float())
print()
print("Interpretation:")
print("- Position 0 can only attend to itself")
print("- Position 1 can attend to positions 0-1")
print("- Position 2 can attend to positions 0-2")
print("- Position 3 can attend to positions 0-3")

In [ ]:
# Test attention with causal masking
attention = ScaledDotProductAttention(d_k, dropout=0.0)
causal_mask = create_causal_mask(seq_len, device).unsqueeze(0)  # Add batch dimension

output, weights = attention(Q, K, V, mask=causal_mask)

print("Attention weights with causal masking:")
print("\nPosition 0 attention weights:")
print(weights[0, 0, :])  # Can only see position 0
print("\nPosition 1 attention weights:")
print(weights[0, 1, :])  # Can see positions 0-1
print("\nPosition 2 attention weights:")
print(weights[0, 2, :])  # Can see positions 0-2
print("\nPosition 3 attention weights:")
print(weights[0, 3, :])  # Can see positions 0-3

## 3.4: Single-Head Attention

In [ ]:
class SingleHeadAttention(nn.Module):
    """Single head of attention with linear projections"""
    
    def __init__(self, d_in, d_out, device=None):
        super().__init__()
        self.d_out = d_out
        
        # Linear projections for Q, K, V
        self.W_query = nn.Linear(d_in, d_out, bias=False, device=device)
        self.W_key = nn.Linear(d_in, d_out, bias=False, device=device)
        self.W_value = nn.Linear(d_in, d_out, bias=False, device=device)
        
        self.attention = ScaledDotProductAttention(d_out, dropout=0.1)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_in) - input sequences
            mask: Optional attention mask
        
        Returns:
            output: (batch, seq_len, d_out)
        """
        # Project to Q, K, V
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)
        
        # Apply attention
        output, weights = self.attention(Q, K, V, mask=mask)
        
        return output, weights

print("SingleHeadAttention class implemented")

## 3.5: Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention mechanism"""
    
    def __init__(self, d_in, d_out, num_heads, device=None):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.num_heads = num_heads
        self.d_out = d_out
        self.head_dim = d_out // num_heads
        
        # Linear projection from d_in to d_out for Q, K, V
        self.W_query = nn.Linear(d_in, d_out, bias=False, device=device)
        self.W_key = nn.Linear(d_in, d_out, bias=False, device=device)
        self.W_value = nn.Linear(d_in, d_out, bias=False, device=device)
        
        # Scaled dot-product attention
        self.attention = ScaledDotProductAttention(self.head_dim, dropout=0.1)
        
        # Final linear projection
        self.W_out = nn.Linear(d_out, d_out, bias=False, device=device)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_in)
            mask: Optional attention mask
        
        Returns:
            output: (batch, seq_len, d_out)
        """
        batch_size = x.shape[0]
        seq_len = x.shape[1]
        
        # 1. Project inputs
        Q = self.W_query(x)  # (batch, seq_len, d_out)
        K = self.W_key(x)
        V = self.W_value(x)
        
        # 2. Reshape for multi-head attention
        # (batch, seq_len, d_out) -> (batch, seq_len, num_heads, head_dim)
        Q = Q.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        V = V.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        
        # 3. Transpose for attention computation
        # (batch, seq_len, num_heads, head_dim) -> (batch, num_heads, seq_len, head_dim)
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        # 4. Apply attention for each head
        attn_output, _ = self.attention(Q, K, V, mask=mask)
        # (batch, num_heads, seq_len, head_dim)
        
        # 5. Concatenate heads
        # (batch, num_heads, seq_len, head_dim) -> (batch, seq_len, num_heads, head_dim)
        attn_output = attn_output.transpose(1, 2)
        # Reshape to (batch, seq_len, d_out)
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_out)
        
        # 6. Final linear projection
        output = self.W_out(attn_output)
        
        return output

print("MultiHeadAttention class implemented")

## 3.6: Testing Multi-Head Attention

In [ ]:
# Create multi-head attention module
d_in = 64
d_out = 64
num_heads = 8

mha = MultiHeadAttention(d_in, d_out, num_heads, device=device).to(device)

# Create input
batch_size = 2
seq_len = 10
x = torch.randn(batch_size, seq_len, d_in, device=device)

# Forward pass without masking
output = mha(x, mask=None)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"\nNumber of heads: {num_heads}")
print(f"Dimension per head: {d_out // num_heads}")
print(f"\nTotal parameters: {sum(p.numel() for p in mha.parameters()):,}")

In [ ]:
# Test with causal masking
causal_mask = create_causal_mask(seq_len, device)
causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)  # Add batch and head dimensions

output_causal = mha(x, mask=causal_mask)

print(f"Output with causal masking shape: {output_causal.shape}")
print(f"\nDifference from non-causal output:")
print(f"  Max difference: {(output - output_causal).abs().max().item():.6f}")
print(f"  Mean difference: {(output - output_causal).abs().mean().item():.6f}")

## 3.7: Attention Efficiency Analysis

In [ ]:
import time

# Measure attention computation time for different sequence lengths
seq_lengths = [32, 64, 128, 256, 512]
times = []

mha = MultiHeadAttention(64, 64, 8, device=device).to(device)

for seq_len in seq_lengths:
    x = torch.randn(16, seq_len, 64, device=device)
    
    # Warm up
    with torch.no_grad():
        _ = mha(x)
    
    # Time the forward pass
    start = time.time()
    with torch.no_grad():
        for _ in range(10):
            _ = mha(x)
    end = time.time()
    
    avg_time = (end - start) / 10 * 1000  # Convert to ms
    times.append(avg_time)
    print(f"Seq length {seq_len:3d}: {avg_time:.2f} ms")

print(f"\nNote: Attention complexity is O(n²) in sequence length")
print(f"Expected: Doubling sequence length should roughly quadruple time")

## 3.8: Summary

### Key Concepts:
1. **Scaled Dot-Product Attention**: Q·K^T / √d_k then softmax then apply to V
2. **Multi-Head Attention**: Run multiple attention heads in parallel and concatenate
3. **Causal Masking**: Prevent attending to future positions (autoregressive)
4. **Complexity**: O(n²) in sequence length due to QK^T computation

### Important Notes:
- Scaling by √d_k stabilizes gradients during backpropagation
- Multiple heads allow attending to different representation subspaces
- Causal masking is essential for language modeling (predicting next token)
- Attention is differentiable, allowing end-to-end training